In [3]:
# Cell 1: Environment and package discovery only
# No model files.
# No raw joblib loading.
# No assumed model call.

from pathlib import Path
import sys
import json
import pkgutil
from importlib import metadata

PROJECT_ROOT = Path(r"C:\Projects\Infer RozviDrought\RozviDrought")
DATA_ROOT = Path(r"C:\Projects\Infer RozviDrought\data")
OUT_DIR = DATA_ROOT / "backtests" / "seed_test"

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("=== Paths ===")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROJECT_ROOT exists:", PROJECT_ROOT.exists())
print("DATA_ROOT:", DATA_ROOT)
print("DATA_ROOT exists:", DATA_ROOT.exists())
print("OUT_DIR:", OUT_DIR)

print("\n=== Python ===")
print("Python executable:", sys.executable)
print("Python version:", sys.version)

print("\n=== Existing notebook objects ===")
print("service exists:", "service" in globals())
print("usable_admin_df exists:", "usable_admin_df" in globals())

print("\n=== Installed distributions matching keywords ===")

keywords = ["rozvi", "drought", "infer"]

matched_distributions = []

for dist in metadata.distributions():
    name = dist.metadata.get("Name", "")
    lower_name = name.lower()

    if any(k in lower_name for k in keywords):
        matched_distributions.append({
            "name": name,
            "version": dist.version,
        })

print(json.dumps(matched_distributions, indent=2))

print("\n=== Importable top-level modules matching keywords ===")

matched_modules = []

for mod in pkgutil.iter_modules():
    lower_name = mod.name.lower()

    if any(k in lower_name for k in keywords):
        matched_modules.append({
            "name": mod.name,
            "is_package": bool(mod.ispkg),
        })

print(json.dumps(matched_modules, indent=2))

context = {
    "project_root": str(PROJECT_ROOT),
    "data_root": str(DATA_ROOT),
    "out_dir": str(OUT_DIR),
    "python_executable": sys.executable,
    "service_exists": "service" in globals(),
    "usable_admin_df_exists": "usable_admin_df" in globals(),
    "matched_distributions": matched_distributions,
    "matched_modules": matched_modules,
}

CONTEXT_JSON = OUT_DIR / "seed_test_environment_context.json"
CONTEXT_JSON.write_text(json.dumps(context, indent=2), encoding="utf-8")

print("\nSaved:", CONTEXT_JSON)

=== Paths ===
PROJECT_ROOT: C:\Projects\Infer RozviDrought\RozviDrought
PROJECT_ROOT exists: True
DATA_ROOT: C:\Projects\Infer RozviDrought\data
DATA_ROOT exists: True
OUT_DIR: C:\Projects\Infer RozviDrought\data\backtests\seed_test

=== Python ===
Python executable: c:\Projects\Infer RozviDrought\.venv\Scripts\python.exe
Python version: 3.12.9 (tags/v3.12.9:fdb8142, Feb  4 2025, 15:27:58) [MSC v.1942 64 bit (AMD64)]

=== Existing notebook objects ===
service exists: False
usable_admin_df exists: False

=== Installed distributions matching keywords ===
[
  {
    "name": "rozvidrought",
    "version": "1.0.6"
  },
  {
    "name": "rozvidrought-datasets",
    "version": "0.0.1"
  },
  {
    "name": "rozvidrought-inputs",
    "version": "0.0.1"
  },
  {
    "name": "rozvidrought-subsystems",
    "version": "0.1.0"
  }
]

=== Importable top-level modules matching keywords ===
[
  {
    "name": "benchmark_inference_latency",
    "is_package": false
  },
  {
    "name": "farm_inference_scrip

In [2]:
# Cell 2: Build synthetic seed master_df for packaged polygon inference
# Uses the real PolygonInferenceService pathway.
# No raw model files.
# No joblib.
# No lower-level assumptions beyond columns used by polygon_inference_service.py
# and the known drought input variables: t2m, d2m, pet, sm, ndvi, tws.

from pathlib import Path
import sys
import pandas as pd
import numpy as np
from shapely.geometry import Polygon

PROJECT_ROOT = Path(r"C:\Projects\Infer RozviDrought\RozviDrought")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.services.polygon_inference_service import PolygonInferenceService

RUN_YYYYMM = 202404
SCENARIO = "historical"
MODEL = "hybrid"

# Synthetic pixels inside a small test polygon
seed_pixels = [
    {"pixel_id": 1, "row": 0, "col": 0, "lon": 30.000, "lat": -18.000},
    {"pixel_id": 2, "row": 0, "col": 1, "lon": 30.010, "lat": -18.000},
    {"pixel_id": 3, "row": 1, "col": 0, "lon": 30.000, "lat": -18.010},
    {"pixel_id": 4, "row": 1, "col": 1, "lon": 30.010, "lat": -18.010},
]

# Enough monthly history for lag/rolling feature preparation
months = pd.period_range("2022-01", "2024-04", freq="M").strftime("%Y%m").tolist()

seed_scenarios = [
    {
        "seed_case": "normal_balanced",
        "expected_class": 0,
        "t2m": 295.0,
        "d2m": 290.0,
        "pet": -0.00025,
        "sm": 0.30,
        "ndvi": 0.65,
        "tws": 3.0,
    },
    {
        "seed_case": "mild_drought",
        "expected_class": 1,
        "t2m": 299.0,
        "d2m": 287.0,
        "pet": -0.00040,
        "sm": 0.23,
        "ndvi": 0.55,
        "tws": -2.0,
    },
    {
        "seed_case": "moderate_drought",
        "expected_class": 2,
        "t2m": 301.0,
        "d2m": 285.0,
        "pet": -0.00052,
        "sm": 0.15,
        "ndvi": 0.40,
        "tws": -8.0,
    },
    {
        "seed_case": "severe_all_systems_dry",
        "expected_class": 3,
        "t2m": 305.0,
        "d2m": 280.0,
        "pet": -0.00075,
        "sm": 0.07,
        "ndvi": 0.22,
        "tws": -18.0,
    },
    {
        "seed_case": "severe_soil_hydrology_only",
        "expected_class": 3,
        "t2m": 296.0,
        "d2m": 288.0,
        "pet": -0.00035,
        "sm": 0.05,
        "ndvi": 0.30,
        "tws": -22.0,
    },
    {
        "seed_case": "hot_but_wet_false_alarm",
        "expected_class": 0,
        "t2m": 305.0,
        "d2m": 293.0,
        "pet": -0.00065,
        "sm": 0.35,
        "ndvi": 0.70,
        "tws": 5.0,
    },
]

rows = []

for case_id, case in enumerate(seed_scenarios, start=1):
    for pixel in seed_pixels:
        synthetic_pixel_id = case_id * 1000 + pixel["pixel_id"]

        for yyyymm in months:
            row = {
                "seed_case": case["seed_case"],
                "expected_class": case["expected_class"],

                "pixel_id": synthetic_pixel_id,
                "row": pixel["row"],
                "col": pixel["col"],
                "lon": pixel["lon"],
                "lat": pixel["lat"],

                "yyyymm": str(yyyymm),
                "scenario": SCENARIO,

                "t2m": case["t2m"],
                "d2m": case["d2m"],
                "pet": case["pet"],
                "sm": case["sm"],
                "ndvi": case["ndvi"],
                "tws": case["tws"],
            }

            rows.append(row)

seed_master_df = pd.DataFrame(rows)

# Polygon covering all synthetic points
seed_polygon = Polygon([
    (29.990, -18.020),
    (30.020, -18.020),
    (30.020, -17.990),
    (29.990, -17.990),
    (29.990, -18.020),
])

seed_service = PolygonInferenceService(master_df=seed_master_df)

print("=== Cell 2: Synthetic seed master_df created ===")
print("Rows:", len(seed_master_df))
print("Pixels:", seed_master_df["pixel_id"].nunique())
print("Months:", seed_master_df["yyyymm"].min(), "to", seed_master_df["yyyymm"].max())
print("Seed cases:", seed_master_df["seed_case"].nunique())

print("\nColumns:")
print(seed_master_df.columns.tolist())

print("\nSeed case summary:")
print(
    seed_master_df
    .groupby(["seed_case", "expected_class"])
    [["t2m", "d2m", "pet", "sm", "ndvi", "tws"]]
    .mean()
    .reset_index()
    .to_string(index=False)
)

=== Cell 2: Synthetic seed master_df created ===
Rows: 672
Pixels: 24
Months: 202201 to 202404
Seed cases: 6

Columns:
['seed_case', 'expected_class', 'pixel_id', 'row', 'col', 'lon', 'lat', 'yyyymm', 'scenario', 't2m', 'd2m', 'pet', 'sm', 'ndvi', 'tws']

Seed case summary:
                 seed_case  expected_class   t2m   d2m      pet   sm  ndvi   tws
   hot_but_wet_false_alarm               0 305.0 293.0 -0.00065 0.35  0.70   5.0
              mild_drought               1 299.0 287.0 -0.00040 0.23  0.55  -2.0
          moderate_drought               2 301.0 285.0 -0.00052 0.15  0.40  -8.0
           normal_balanced               0 295.0 290.0 -0.00025 0.30  0.65   3.0
    severe_all_systems_dry               3 305.0 280.0 -0.00075 0.07  0.22 -18.0
severe_soil_hydrology_only               3 296.0 288.0 -0.00035 0.05  0.30 -22.0


In [4]:
# Cell 3: Inspect one seed case through feature preparation and subsystems
# Purpose:
# - Do not run fusion yet.
# - Find why hydrology output is empty.
# - Use one seed case / one location path.

TEST_SEED_CASE = "severe_all_systems_dry"
TEST_PIXEL_ID = seed_master_df.loc[
    seed_master_df["seed_case"].eq(TEST_SEED_CASE),
    "pixel_id"
].iloc[0]

case_df = seed_master_df[
    seed_master_df["seed_case"].eq(TEST_SEED_CASE)
].copy()

case_service = PolygonInferenceService(master_df=case_df)

ts = case_service._prepare_timeseries(
    pixel_id=int(TEST_PIXEL_ID),
    scenario="historical",
    run_yyyymm=202404,
)

prepared = case_service.feature_service.prepare_subsystem_inputs(
    ts,
    run_yyyymm=202404,
)

subsystem_outputs = case_service.subsystem_service.run_subsystems(prepared)

print("=== Cell 3: One-case subsystem inspection ===")
print("Seed case:", TEST_SEED_CASE)
print("Pixel ID:", TEST_PIXEL_ID)
print("Timeseries rows:", len(ts))
print("Timeseries months:", ts["yyyymm"].min(), "to", ts["yyyymm"].max())

print("\nTimeseries columns:")
print(ts.columns.tolist())

print("\nPrepared subsystem inputs:")
for name, df in prepared.items():
    print("\n", name)
    print("shape:", df.shape)
    print("columns:", df.columns.tolist())
    print(df.tail(3).to_string(index=False))

print("\nSubsystem outputs:")
for name, df in subsystem_outputs.items():
    print("\n", name)
    print("shape:", df.shape)
    print("columns:", df.columns.tolist() if not df.empty else [])
    print(df.tail(3).to_string(index=False) if not df.empty else "EMPTY")

=== Cell 3: One-case subsystem inspection ===
Seed case: severe_all_systems_dry
Pixel ID: 4001
Timeseries rows: 28
Timeseries months: 202201 to 202404

Timeseries columns:
['seed_case', 'expected_class', 'pixel_id', 'row', 'col', 'lon', 'lat', 'yyyymm', 'scenario', 't2m', 'd2m', 'pet', 'sm', 'ndvi', 'tws']

Prepared subsystem inputs:

 atmospheric
shape: (28, 28)
columns: ['pixel_key', 'yyyymm', 'pet', 't2m', 'd2m', 'pet_lag1', 'pet_lag2', 'pet_lag3', 'pet_lag6', 't2m_lag1', 't2m_lag2', 't2m_lag3', 't2m_lag6', 'd2m_lag1', 'd2m_lag2', 'd2m_lag3', 'd2m_lag6', 'pet_rollmean3', 'pet_rollmean6', 'pet_rollmean12', 't2m_rollmean3', 't2m_rollmean6', 't2m_rollmean12', 'd2m_rollmean3', 'd2m_rollmean6', 'd2m_rollmean12', 'time_type', 'run_yyyymm']
 pixel_key yyyymm      pet   t2m   d2m  pet_lag1  pet_lag2  pet_lag3  pet_lag6  t2m_lag1  t2m_lag2  t2m_lag3  t2m_lag6  d2m_lag1  d2m_lag2  d2m_lag3  d2m_lag6  pet_rollmean3  pet_rollmean6  pet_rollmean12  t2m_rollmean3  t2m_rollmean6  t2m_rollmean12  d

In [5]:
# Cell 4: Test whether hydrology needs longer time history

import pandas as pd
import numpy as np

assert "PolygonInferenceService" in globals(), "Run Cell 2 first."
assert "seed_scenarios" in globals(), "Run Cell 2 first."
assert "seed_pixels" in globals(), "Run Cell 2 first."

RUN_YYYYMM = 202404
SCENARIO = "historical"
TEST_SEED_CASE = "severe_all_systems_dry"

case = next(x for x in seed_scenarios if x["seed_case"] == TEST_SEED_CASE)
pixel = seed_pixels[0]

history_tests = [
    ("2022-01", "28_months"),
    ("2021-01", "40_months"),
    ("2020-01", "52_months"),
    ("2019-01", "64_months"),
    ("2018-01", "76_months"),
]

rows = []

for start_month, test_name in history_tests:
    months = pd.period_range(start_month, "2024-04", freq="M").strftime("%Y%m").tolist()

    test_pixel_id = 9001

    test_rows = []

    for yyyymm in months:
        test_rows.append({
            "seed_case": case["seed_case"],
            "expected_class": case["expected_class"],

            "pixel_id": test_pixel_id,
            "row": pixel["row"],
            "col": pixel["col"],
            "lon": pixel["lon"],
            "lat": pixel["lat"],

            "yyyymm": str(yyyymm),
            "scenario": SCENARIO,

            "t2m": case["t2m"],
            "d2m": case["d2m"],
            "pet": case["pet"],
            "sm": case["sm"],
            "ndvi": case["ndvi"],
            "tws": case["tws"],
        })

    test_master_df = pd.DataFrame(test_rows)

    test_service = PolygonInferenceService(master_df=test_master_df)

    ts = test_service._prepare_timeseries(
        pixel_id=test_pixel_id,
        scenario=SCENARIO,
        run_yyyymm=RUN_YYYYMM,
    )

    prepared = test_service.feature_service.prepare_subsystem_inputs(
        ts,
        run_yyyymm=RUN_YYYYMM,
    )

    subsystem_outputs = test_service.subsystem_service.run_subsystems(prepared)

    hyd_input = prepared.get("hydrology")
    hyd_output = subsystem_outputs.get("hydrology")

    rows.append({
        "test_name": test_name,
        "start_month": start_month,
        "ts_rows": len(ts),
        "hydrology_input_rows": len(hyd_input) if hyd_input is not None else None,
        "hydrology_output_rows": len(hyd_output) if hyd_output is not None else None,
        "hydrology_output_empty": bool(hyd_output is None or hyd_output.empty),
    })

    print("\n" + "=" * 80)
    print("History test:", test_name)
    print("Start month:", start_month)
    print("Timeseries rows:", len(ts))
    print("Hydrology input shape:", hyd_input.shape if hyd_input is not None else None)
    print("Hydrology output shape:", hyd_output.shape if hyd_output is not None else None)

    if hyd_output is not None and not hyd_output.empty:
        print("\nHydrology output tail:")
        print(hyd_output.tail(5).to_string(index=False))

hyd_history_df = pd.DataFrame(rows)

OUT_CSV = OUT_DIR / "seed_test_hydrology_history_requirement.csv"
hyd_history_df.to_csv(OUT_CSV, index=False)

print("\n=== Cell 4: Hydrology history test summary ===")
print(hyd_history_df.to_string(index=False))

print("\nSaved:", OUT_CSV)


History test: 28_months
Start month: 2022-01
Timeseries rows: 28
Hydrology input shape: (28, 12)
Hydrology output shape: (0, 4)

History test: 40_months
Start month: 2021-01
Timeseries rows: 40
Hydrology input shape: (40, 12)
Hydrology output shape: (1, 4)

Hydrology output tail:
      p0       p1       p2           p3
0.049946 0.946536 0.003518 2.349732e-07

History test: 52_months
Start month: 2020-01
Timeseries rows: 52
Hydrology input shape: (52, 12)
Hydrology output shape: (1, 4)

Hydrology output tail:
      p0       p1       p2           p3
0.049946 0.946536 0.003518 2.349732e-07

History test: 64_months
Start month: 2019-01
Timeseries rows: 64
Hydrology input shape: (64, 12)
Hydrology output shape: (1, 4)

Hydrology output tail:
      p0       p1       p2           p3
0.049946 0.946536 0.003518 2.349732e-07

History test: 76_months
Start month: 2018-01
Timeseries rows: 76
Hydrology input shape: (76, 12)
Hydrology output shape: (1, 4)

Hydrology output tail:
      p0       p1  

NameError: name 'OUT_DIR' is not defined

In [6]:
# Cell 5: One-location seed test across all classes using 40-month history

from pathlib import Path
import pandas as pd
import numpy as np
from shapely.geometry import Polygon

OUT_DIR = Path(r"C:\Projects\Infer RozviDrought\data\backtests\seed_test")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_YYYYMM = 202404
SCENARIO = "historical"
MODEL = "hybrid"

months = pd.period_range("2021-01", "2024-04", freq="M").strftime("%Y%m").tolist()

# One random Zimbabwe-like test location
test_pixel = {
    "pixel_id": 10001,
    "row": 0,
    "col": 0,
    "lon": 30.20,
    "lat": -19.00,
}

seed_polygon = Polygon([
    (30.19, -19.01),
    (30.21, -19.01),
    (30.21, -18.99),
    (30.19, -18.99),
    (30.19, -19.01),
])

seed_cases = [
    {
        "seed_case": "class_0_normal_wet",
        "expected_class": 0,
        "t2m": 294.0,
        "d2m": 290.0,
        "pet": -0.00020,
        "sm": 0.34,
        "ndvi": 0.70,
        "tws": 6.0,
    },
    {
        "seed_case": "class_0_hot_but_wet",
        "expected_class": 0,
        "t2m": 305.0,
        "d2m": 294.0,
        "pet": -0.00065,
        "sm": 0.36,
        "ndvi": 0.72,
        "tws": 7.0,
    },
    {
        "seed_case": "class_1_mild_dry",
        "expected_class": 1,
        "t2m": 298.0,
        "d2m": 287.0,
        "pet": -0.00038,
        "sm": 0.23,
        "ndvi": 0.55,
        "tws": -2.0,
    },
    {
        "seed_case": "class_1_mild_hydrology_stress",
        "expected_class": 1,
        "t2m": 296.0,
        "d2m": 288.0,
        "pet": -0.00032,
        "sm": 0.22,
        "ndvi": 0.52,
        "tws": -5.0,
    },
    {
        "seed_case": "class_2_moderate_dry",
        "expected_class": 2,
        "t2m": 301.0,
        "d2m": 285.0,
        "pet": -0.00052,
        "sm": 0.15,
        "ndvi": 0.40,
        "tws": -9.0,
    },
    {
        "seed_case": "class_2_moderate_vegetation_stress",
        "expected_class": 2,
        "t2m": 299.0,
        "d2m": 286.0,
        "pet": -0.00048,
        "sm": 0.17,
        "ndvi": 0.32,
        "tws": -8.0,
    },
    {
        "seed_case": "class_3_severe_all_systems",
        "expected_class": 3,
        "t2m": 305.0,
        "d2m": 280.0,
        "pet": -0.00075,
        "sm": 0.07,
        "ndvi": 0.22,
        "tws": -18.0,
    },
    {
        "seed_case": "class_3_severe_soil_hydrology",
        "expected_class": 3,
        "t2m": 296.0,
        "d2m": 288.0,
        "pet": -0.00035,
        "sm": 0.05,
        "ndvi": 0.30,
        "tws": -22.0,
    },
]

def extract_fusion_prediction(cell_result):
    result = cell_result.get("result")

    if isinstance(result, dict) and "fusion_result" in result:
        result = result["fusion_result"]

    if hasattr(result, "drought_class_spi3_pred"):
        pred = np.asarray(result.drought_class_spi3_pred).ravel()
        conf = np.asarray(result.confidence).ravel() if hasattr(result, "confidence") else []

        pred_val = int(pred[0]) if len(pred) else None
        conf_val = float(conf[0]) if len(conf) else None

        return pred_val, conf_val

    return None, None

results = []

for case in seed_cases:
    rows = []

    for yyyymm in months:
        rows.append({
            "seed_case": case["seed_case"],
            "expected_class": case["expected_class"],

            "pixel_id": test_pixel["pixel_id"],
            "row": test_pixel["row"],
            "col": test_pixel["col"],
            "lon": test_pixel["lon"],
            "lat": test_pixel["lat"],

            "yyyymm": str(yyyymm),
            "scenario": SCENARIO,

            "t2m": case["t2m"],
            "d2m": case["d2m"],
            "pet": case["pet"],
            "sm": case["sm"],
            "ndvi": case["ndvi"],
            "tws": case["tws"],
        })

    case_master_df = pd.DataFrame(rows)
    case_service = PolygonInferenceService(master_df=case_master_df)

    try:
        result = case_service.infer_polygon(
            geometry=seed_polygon,
            scenario=SCENARIO,
            yyyymm=RUN_YYYYMM,
            model=MODEL,
        )

        preds = []
        confs = []

        for cell_result in result.cell_results:
            pred_val, conf_val = extract_fusion_prediction(cell_result)

            if pred_val is not None:
                preds.append(pred_val)

            if conf_val is not None:
                confs.append(conf_val)

        predicted_class = int(pd.Series(preds).mode().iloc[0]) if preds else None
        mean_confidence = float(np.mean(confs)) if confs else None

        status = "success"
        error = None

    except Exception as exc:
        predicted_class = None
        mean_confidence = None
        status = "failed"
        error = f"{type(exc).__name__}: {exc}"

    results.append({
        "seed_case": case["seed_case"],
        "expected_class": case["expected_class"],
        "predicted_class": predicted_class,
        "correct": predicted_class == case["expected_class"] if predicted_class is not None else False,
        "mean_confidence": mean_confidence,
        "status": status,
        "error": error,

        "t2m": case["t2m"],
        "d2m": case["d2m"],
        "vpd_proxy": case["t2m"] - case["d2m"],
        "pet": case["pet"],
        "sm": case["sm"],
        "ndvi": case["ndvi"],
        "tws": case["tws"],
    })

seed_result_df = pd.DataFrame(results)

OUT_CSV = OUT_DIR / "seed_test_one_location_all_classes.csv"
seed_result_df.to_csv(OUT_CSV, index=False)

print("=== Cell 5: One-location seed test across all classes ===")
print(seed_result_df.to_string(index=False))

valid_df = seed_result_df[seed_result_df["status"].eq("success")].copy()

print("\nRows:", len(seed_result_df))
print("Successful:", len(valid_df))

if len(valid_df):
    print("Seed accuracy:", round(float(valid_df["correct"].mean()), 4))

    print("\nConfusion table:")
    print(
        pd.crosstab(
            valid_df["expected_class"],
            valid_df["predicted_class"],
            rownames=["expected"],
            colnames=["predicted"],
            dropna=False,
        )
    )

print("\nSaved:", OUT_CSV)


=== Cell 5: One-location seed test across all classes ===
                         seed_case  expected_class  predicted_class  correct  mean_confidence  status error   t2m   d2m  vpd_proxy      pet   sm  ndvi   tws
                class_0_normal_wet               0                0     True         0.888635 success  None 294.0 290.0        4.0 -0.00020 0.34  0.70   6.0
               class_0_hot_but_wet               0                0     True         0.561369 success  None 305.0 294.0       11.0 -0.00065 0.36  0.72   7.0
                  class_1_mild_dry               1                1     True         0.545850 success  None 298.0 287.0       11.0 -0.00038 0.23  0.55  -2.0
     class_1_mild_hydrology_stress               1                0    False         0.763533 success  None 296.0 288.0        8.0 -0.00032 0.22  0.52  -5.0
              class_2_moderate_dry               2                0    False         0.568613 success  None 301.0 285.0       16.0 -0.00052 0.15  0.40  -9.0